In [1]:
import sys
sys.path.append("../")

from models_training.train_segmentation import (
    create_spark,
    load_data,
    cast_numeric_columns,
    assemble_features,
    scale_features,
    train_kmeans,
    evaluate_model,
    save_segmented_data,
    save_model
)

In [2]:
# creer spark
spark = create_spark()

26/03/06 16:57:05 WARN Utils: Your hostname, nouhayla-HP-EliteBook-835-G7-Notebook-PC resolves to a loopback address: 127.0.1.1; using 192.168.8.126 instead (on interface wlp1s0)
26/03/06 16:57:05 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/06 16:57:06 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/06 16:57:07 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [3]:
# load data
df = load_data(spark, "../data/silver/silver_data")
df.show(5)

+----------+---+------+------+---------------+------------+------------------+-------------------+-------------+------------------+------------------+------------+----------+-----------+-----------------+-------------+-------------------+---------------+----------+
|CustomerID|Age|Gender|Income|CampaignChannel|CampaignType|           AdSpend|   ClickThroughRate|WebsiteVisits|     PagesPerVisit|        TimeOnSite|SocialShares|EmailOpens|EmailClicks|PreviousPurchases|LoyaltyPoints|AdvertisingPlatform|AdvertisingTool|Conversion|
+----------+---+------+------+---------------+------------+------------------+-------------------+-------------+------------------+------------------+------------+----------+-----------+-----------------+-------------+-------------------+---------------+----------+
|      8000| 56|Female|136912|   Social Media|   Awareness| 6497.870068417766|0.04391851073538301|            0| 2.399016527783845|7.3968025807960585|          19|         6|          9|                

In [4]:
df = cast_numeric_columns(df)
df.printSchema()

root
 |-- CustomerID: string (nullable = true)
 |-- Age: double (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Income: double (nullable = true)
 |-- CampaignChannel: string (nullable = true)
 |-- CampaignType: string (nullable = true)
 |-- AdSpend: string (nullable = true)
 |-- ClickThroughRate: string (nullable = true)
 |-- WebsiteVisits: string (nullable = true)
 |-- PagesPerVisit: string (nullable = true)
 |-- TimeOnSite: string (nullable = true)
 |-- SocialShares: string (nullable = true)
 |-- EmailOpens: string (nullable = true)
 |-- EmailClicks: string (nullable = true)
 |-- PreviousPurchases: double (nullable = true)
 |-- LoyaltyPoints: double (nullable = true)
 |-- AdvertisingPlatform: string (nullable = true)
 |-- AdvertisingTool: string (nullable = true)
 |-- Conversion: string (nullable = true)



In [5]:
df = assemble_features(df)
df.select("features").show(5, truncate=False)

+-------------------------+
|features                 |
+-------------------------+
|[56.0,136912.0,688.0,4.0]|
|[69.0,41760.0,3459.0,2.0]|
|[46.0,88456.0,2337.0,8.0]|
|[32.0,44085.0,2463.0,0.0]|
|[60.0,83964.0,4345.0,8.0]|
+-------------------------+
only showing top 5 rows



In [6]:
# normalisation
df = scale_features(df)
df.select("scaled_features").show(5, truncate=False)


+----------------------------------------------------------------------------------+
|scaled_features                                                                   |
+----------------------------------------------------------------------------------+
|[0.8303481331934817,1.3902944090407785,-1.2607444951158608,-0.16810399278166432]  |
|[1.7026682860493738,-1.1416645515313792,0.6776586872990041,-0.8606024182467842]   |
|[0.15933263099664144,0.10089845947078022,-0.10721622091805773,1.2168928581485754] |
|[-0.7800890720789347,-1.0797971752982967,-0.019075188444483946,-1.553100843711904]|
|[1.0987543340722177,-0.018631972373953802,1.2974440426290548,1.2168928581485754]  |
+----------------------------------------------------------------------------------+
only showing top 5 rows



In [7]:
# train model
model = train_kmeans(df, k=4)

26/03/06 16:57:15 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.VectorBLAS


In [8]:
# metrics
metric = evaluate_model(model, df)
print("Silhouette score:", metric)

Silhouette score: 0.31373381458619226


In [9]:
# sauvgarde du new data 
save_segmented_data(model, df, "../data/silver/segment_data")

In [10]:
# save model
model.write().overwrite().save("models/segmentation_model")

print("segment model trained successfully")

segment model trained successfully


26/03/06 16:57:18 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors
